**PRÁCTICA 4: DETECCIÓN Y LECTURA DE MATRÍCULAS CON YOLO Y PADDLEOCR**

En este cuaderno se desarrolla el sistema de detección, lectura y seguimiento de matrículas mediante modelos de visión por computador y reconocimiento óptico de caracteres (OCR).

El flujo de trabajo se estructura en:
1. **Configuración inicial e imports necesarios**: Carga de librerías comunes y definición global de los modelos de detección y OCR.
2. **Utilidades comunes y función unificada de lectura**: Operaciones geométricas, métricas de similitud de cadenas y la función única `leer_placa`.
3. **Adquisición de métricas de OCR**: Evaluación cuantitativa del sistema sobre el conjunto de datos de placas (extremo a extremo vs recorte de anotación).
4. **Lectura y seguimiento en vídeo**: Tracking de vehículos y peatones mediante ByteTrack, asociación y filtrado temporal de placas, y renderizado del vídeo final.


**IMPORTS NECESARIOS Y CONFIGURACIÓN GLOBAL**

Para llevar a cabo la detección y lectura de matrículas, inicialmente se importan las librerías necesarias y se configuran las dependencias del sistema.


In [1]:
import os
from pathlib import Path

from ultralytics import YOLO

**INICIALIZACIÓN GLOBAL DE MODELOS**

Definimos una única instancia global para el detector de matrículas entrenado (`best-yolo.pt`) y para el motor de reconocimiento PaddleOCR. De esta forma evitamos recargar modelos en memoria o duplicar inicializaciones entre la adquisición de métricas y el procesado de vídeo.


In [2]:
# Ruta al detector entrenado de placas
WEIGHTS_PLACAS = "weights/best-yolo.pt"

# Inicialización del detector de matrículas compartido
detector_placas = YOLO(WEIGHTS_PLACAS)

# Motor OCR global (definido en utils_matriculas.py)
from utils_matriculas import get_ocr_paddle as get_ocr

# Inicializamos el OCR globalmente
ocr_global = get_ocr()

c:\Users\vidal\miniconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\vidal\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
c:\Users\vidal\miniconda3\envs\VC_P4\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\vidal\.padd

**UTILIDADES COMUNES: GEOMETRÍA, CADENAS Y MANIPULACIÓN DE CAJAS**

A continuación se definen las funciones auxiliares compartidas necesarias para el tratamiento de texto, cálculo de solapamiento de cajas (IoU), cálculo de centros y distancias, y recorte de regiones de interés.


In [3]:
from utils_matriculas import norm, lev, iou, centro, diag, recortar

**FUNCIÓN ÚNICA DE DETECCIÓN Y LECTURA DE MATRÍCULAS (`leer_placa`)**

Implementa la lectura OCR de la matrícula a partir del recorte de imagen. Combina un preprocesamiento que asegura dimensiones mínimas mediante interpolación bicúbica, filtrado Gaussiano y padding réplica, para posteriormente ordenar espacialmente el texto detectado (soportando placas de una o dos líneas) y calcular la confianza media de lectura.


In [4]:
from utils_matriculas import leer_placa_paddle as leer_placa

**TAREA 1: ADQUISICIÓN Y EVALUACIÓN DE MÉTRICAS OCR**

En esta fase se evalúa el rendimiento cuantitativo del sistema sobre el conjunto de anotaciones reales (Ground Truth):
- **Exact Match**: Porcentaje de matrículas donde la cadena leída coincide al 100% con el texto real.
- **Acierto por carácter**: Precisión media basada en la distancia normalizada de Levenshtein: $1 - \min(1, \text{lev} / \text{longitud})$.
- **Tiempos de ejecución (`perf_counter`)**: Se mide el tiempo medio empleado por detección tanto en localización (YOLO) como en reconocimiento (PaddleOCR), así como el coste temporal total del pipeline por detección.
- **Análisis de falsos positivos**: Se diferencian las detecciones sin coincidencia con ninguna caja real (falsos positivos geométricos) de aquellas que coinciden con una placa anotada pero sin texto legible (`None`/vacía).


In [5]:
# Configuración de rutas y parámetros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
GT_KEYS = ("lpn", "text", "plate", "label", "number", "plate_number")
CONF_METRICAS = 0.25
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("ocr_eval")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "errores-yolo").mkdir(exist_ok=True)


from utils_matriculas import cargar_gt as _cargar_gt, resumen_metricas

# Carga las cajas y textos anotados desde los archivos JSON
def cargar_gt(img_path, w, h):
    return _cargar_gt(img_path, w, h, ANOTACIONES, GT_KEYS)


A continuación se implementa la función `evaluar_dataset`. Emplea `perf_counter` para registrar con alta precisión los tiempos medios de lectura OCR por detección (tanto a partir de las detecciones de YOLO como a partir de los recortes anotados del Ground Truth), clasifica rigurosamente los posibles falsos positivos geométricos frente a placas anotadas con texto `None`, y genera tanto el archivo `ocr_resultados.csv` como el guardado de recortes erróneos en disco.


In [6]:
from utils_matriculas import evaluar_dataset as _evaluar_dataset


def evaluar_dataset(lista_imagenes, detector=detector_placas):
    """Evalúa OCR en detecciones YOLO y en recortes anotados (techo OCR), midiendo tiempos por detección con perf_counter."""
    def detectar(img):
        r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]
        return [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]

    return _evaluar_dataset(lista_imagenes, detectar, leer_placa, cargar_gt, SALIDA, "errores-yolo",
                            "yolo_resultados_ocr.csv", "YOLO", iou_match=IOU_MATCH, pad=PAD,
                            gt_keys=GT_KEYS, delimiter=";")


# Ejecución de la evaluación de métricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")



Imágenes evaluadas: 200 | placas anotadas: 407 (con texto: 407) | detecciones YOLO: 367
--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---
OCR a partir de detección YOLO:     501.84 ms / detección
OCR a partir de recorte GT (techo): 629.43 ms / recorte anotado

--- RESUMEN DE EXACTITUD OCR ---
OCR sobre detecciones de YOLO      n=188   exact match=0.734 | acierto por carácter=0.894
OCR sobre recorte anotado (techo)  n=210   exact match=0.648 | acierto por carácter=0.791
Placas evaluadas encontradas por YOLO: 188/407
Falsos positivos geométricos (IoU < 0.5): 108
Detecciones en placas anotadas sin texto (None): 71

CSV: ocr_eval\yolo_resultados_ocr.csv
Recortes con error: ocr_eval\errores-yolo


**TAREA 2: DETECCIÓN, TRACKING Y LECTURA DE MATRÍCULAS EN VÍDEO**

En este módulo se procesa una secuencia de vídeo realizando:
- **Seguimiento con ByteTrack**: Detección y seguimiento continuo de personas y vehículos (`persona`, `coche`, `moto`, `guagua`, `camion`).
- **Filtrado y validación sintáctica**: Verificación de las matrículas leídas mediante expresiones regulares de formatos de placas europeos.
- **Fusión de tracklets**: Recuperación de identificadores ante oclusiones temporales, tanto para vehículos en movimiento como para vehículos estacionados.
- **Renderizado y exportación**: Generación del vídeo anotado con los identificadores y recuadros de matrícula, junto con el archivo `resultado.csv` detallando las detecciones por fotograma.


In [7]:
# Configuración de rutas de vídeo
filename = "C0142.MP4"
carpeta = os.path.join(os.path.dirname(os.path.abspath(filename)), "YOLO_PaddleOCR_result")
os.makedirs(carpeta, exist_ok=True)
output_path = os.path.join(carpeta, "resultado.mp4")
csv_path = os.path.join(carpeta, "resultado.csv")

# Mapeo de clases y paleta visual
nombre = {0: "persona", 2: "coche", 3: "moto", 5: "guagua", 7: "camion"}
colores = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255),
           5: (0, 255, 255), 7: (255, 255, 0)}

COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35
CONF_PLACA = 0.4
MIN_FRAMES_PLACA = 5

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

# Patrones sintácticos de matrículas europeas (definidos en utils_matriculas.py)
from utils_matriculas import PATRONES_MATRICULA

# Parámetros de tracking y fusión de tracklets
# (FAMILIA, PARAM_FUSION, GAP_ESTATICO_SEG, MIN_FRAMES_ESTATICO, IOU_ESTATICO y MAYORIA_PESADOS
#  se definen en utils_matriculas.py, donde los usan fusionar_tracklets y decidir_clase)
MIN_FRAMES = 10
PESO_BORDE = 0.3
MARGEN_BORDE = 5

# Configuración del archivo de ByteTrack
os.makedirs("./YOLO", exist_ok=True)
with open("./YOLO/mi_bytetrack.yaml", "w") as f:
    f.write("tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.1\n"
            "new_track_thresh: 0.25\ntrack_buffer: 140\nmatch_thresh: 0.8\nfuse_score: True\n")


Finalmente, se define la función `procesar_video` que integra las dos pasadas:
1. **Pasada 1**: Ejecución de inferencia sobre el vídeo, seguimiento con ByteTrack, detección de matrículas mediante `detector_placas`, lectura mediante `leer_placa` y fusión temporal de identidades.
2. **Pasada 2**: Renderizado del vídeo final (`resultado.mp4`) superponiendo las cajas y matrículas leídas, y volcado a `resultado.csv`.


In [10]:
# Ejecución del pipeline de vídeo
from utils_matriculas import procesar_video_yolo, leer_placa_paddle

# Ejecución del pipeline de vídeo
if os.path.exists(filename):
    procesar_video_yolo(
        filename, output_path, csv_path, detector_placas, leer_placa, nombre, colores,
        conf_placa=CONF_PLACA, min_frames_placa=MIN_FRAMES_PLACA, color_placa=COLOR_PLACA,
        escala_texto_placa=ESCALA_TEXTO_PLACA, ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR,
        min_chars_placa=MIN_CHARS_PLACA, ocr_score_ok=OCR_SCORE_OK,
        ocr_max_intentos=OCR_MAX_INTENTOS, min_frames=MIN_FRAMES,
        peso_borde=PESO_BORDE, margen_borde=MARGEN_BORDE,
        track_kwargs=dict(quantize=16, verbose=False),
        predict_kwargs=dict(quantize=16, verbose=False),
        etiqueta_ocr="VLM", ocr_solo_en_placas=True,
        max_ancho_salida=1920, mostrar_progreso=True,
    )
else:
    print(f"Vídeo {filename} no encontrado. Comprueba la ruta.")


Iniciando Pasada 1: tracking, detección y lectura OCR en vídeo...
IDs originales: 187 -> tras fusionar: 95
Iniciando Pasada 2: renderizado de vídeo y escritura en CSV...
Video guardado en c:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\YOLO_PaddleOCR_result\resultado.mp4 | CSV guardado en c:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\YOLO_PaddleOCR_result\resultado.csv
Matrículas detectadas: 40 | Matrículas leídas con OCR: 23
